# Õppetund 09 - Metakognitsiooni disainimuster


## Seadistamine

See märkmik demonstreerib metakognitsiooni disainimustrit, kasutades Microsoft Agent Frameworki.

**Eeltingimused:**
- Azure OpenAI juurutus on konfigureeritud keskkonnamuutujate kaudu
- Azure CLI on autentitud (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mis on metakognitsioon?

Metakognitsioon on **mõtlemise üle mõtlemine**. Tehisintellekti agentide kontekstis tähendab see agentide loomist, kes suudavad:

- **Enda üle mõtiskleda** oma väljundite ja järeldusprotsessi osas
- **Vigu tuvastada** ja taastuda sujuvalt selle asemel, et vaikselt läbi kukkuda
- **Hinnata**, kas nende vastused on täielikud ja kasulikud
- **Kohandada** oma strateegiat, kui esialgne lähenemine ei toimi (nt pöördudes varusüsteemi poole)

Metakognitiivne agent ei vasta ainult küsimustele — ta jälgib oma sooritust ja kohandub jooksvalt.


## Põhi- ja varutööriistad

Levinud metakognitsiooni muster on **tagavarastrateegia**. Agent proovib esmalt peamist tööriista; kui see ebaõnnestub (nt 404 viga), tunneb agent vea ära ja lülitub läbipaistvalt üle varutööriistale.

See peegeldab reaalse maailma süsteeme, kus põhiteenused võivad olla kättesaamatud ning agendid peavad enne alternatiivse tee valimist probleemi iseseisvalt diagnoosima.

Allpool määratleme kaks lennuotsingu tööriista:
- **Peamine** — katab Pariisi, Tokiot ja Barcelonat
- **Varu** — katab Berliini, Sydneyt ja New Yorgi


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Enesereflekteeriv agent vigade taastamisega

Allolev agent on juhendatud esmalt proovima peamist lennusüsteemi, tuvastama tõrkeid ja läbipaistvalt lülituma tagavarasüsteemi. Pärast iga vastust hindab ta lühidalt, kas ta vastas kasutaja küsimusele täielikult.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Enesehindamise muster

Teine metakognitsiooni tahk on **enesehindamine**: eraldi agent (või sama agent teisel läbivaatusel) vaatab vastuse üle täielikkuse, täpsuse ja kasulikkuse osas.

Allpool loome `ResponseEvaluator` agendi, kes hindab reisiagendi vastuseid kolmel mõõdetel.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Kokkuvõte

Selles õppetükis õppisite, kuidas ehitada **metakognitiivseid agente**, kasutades Microsoft Agent Frameworki:

- **Enesereflektsioon**: Agendid, kes jälgivad oma põhjendusi ja teavitavad läbipaistvalt toimunust.
- **Vea taastumine tagavaralahendustega**: Põhi- ja varutööriistamuster, kus agent tuvastab tõrkeid (nt 404 vead) ja proovib automaatselt alternatiivset allikat.
- **Enesehindamine**: Eraldi hindajaagent, mis hindab vastuseid terviklikkuse, täpsuse ja kasulikkuse alusel.

Need mustrid muudavad agendid vastupidavamaks, läbipaistvamaks ja usaldusväärsemaks — kriitilised omadused tootmisse juurutamiseks.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Lahtiütlus:
See dokument on tõlgitud tehisintellekti tõlketeenuse Co-op Translator (https://github.com/Azure/co-op-translator) abil. Kuigi me püüdleme täpsuse poole, tuleb arvestada, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokumenti selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste ega valesti tõlgendamise eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
